# Line Following — Part 2: Live Robot Control

This notebook runs the trained CNN on live camera frames and controls the robot.

**Behaviour:**
- `forward` prediction → move forward at cruise speed
- `left` prediction    → slow down and turn left
- `right` prediction   → slow down and turn right
- Line **lost** (low confidence for several frames) → stop and spin slowly to search

**Prerequisites:** Run `Line_Following_1_Train.ipynb` first to generate `line_follower.pth`.

## Step 1 — Load the Trained Model

In [3]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn

MODEL_PATH = 'line_follower.pth'
IMG_SIZE   = 224

# Class order is alphabetical (how ImageFolder loads them)
# Verify this matches what was printed during training!
CLASS_NAMES = ['forward', 'left', 'right']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on: {device}")

# ── Rebuild the same architecture as training ─────────────────────────────────
model = torchvision.models.mobilenet_v2(weights=None)
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.last_channel, 3)
)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()
print("Model loaded successfully.")

# Image pre-processing pipeline (must match training)
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

Running on: cuda


/tmp/ipykernel_14118/398345579.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_PATH, map_location=device))


Model loaded successfully.


## Step 2 — Start the Camera and Robot

In [ ]:
import traitlets
import cv2
import numpy as np
import pyzed.sl as sl
import threading
import time
import motors
from traitlets.config.configurable import SingletonConfigurable

# ── Speed / behaviour parameters ─────────────────────────────────────────────
# Adjust these to tune performance on your specific track.
SPEED_FORWARD   = 0.35   # cruise speed on straight sections
SPEED_TURN      = 0.30   # speed during turns (slightly slower)
SPEED_SEARCH    = 0.25   # spin speed when line is lost
CONF_THRESHOLD  = 0.55   # minimum confidence to trust a prediction
LOST_FRAMES_MAX = 8      # frames of low confidence before declaring line lost
# ──────────────────────────────────────────────────────────────────────────────

class Camera(SingletonConfigurable):
    color_value = traitlets.Any()

    def __init__(self):
        super(Camera, self).__init__()
        self.zed = sl.Camera()
        init_params = sl.InitParameters()
        init_params.camera_resolution = sl.RESOLUTION.VGA
        init_params.depth_mode = sl.DEPTH_MODE.NONE
        init_params.coordinate_units = sl.UNIT.MILLIMETER
        status = self.zed.open(init_params)
        if status != sl.ERROR_CODE.SUCCESS:
            print("Camera Open:", repr(status))
            self.zed.close()
            exit(1)
        self.runtime = sl.RuntimeParameters()
        self.thread_runnning_flag = False
        camera_info = self.zed.get_camera_information()
        self.width  = camera_info.camera_configuration.resolution.width
        self.height = camera_info.camera_configuration.resolution.height
        self.image  = sl.Mat(self.width, self.height, sl.MAT_TYPE.U8_C4, sl.MEM.CPU)

    def _capture_frames(self):
        while self.thread_runnning_flag:
            if self.zed.grab(self.runtime) == sl.ERROR_CODE.SUCCESS:
                self.zed.retrieve_image(self.image, sl.VIEW.LEFT)
                bgra = self.image.get_data()
                self.color_value = cv2.cvtColor(bgra, cv2.COLOR_BGRA2BGR)

    def start(self):
        if not self.thread_runnning_flag:
            self.thread_runnning_flag = True
            self.thread = threading.Thread(target=self._capture_frames)
            self.thread.start()

    def stop(self):
        if self.thread_runnning_flag:
            self.thread_runnning_flag = False
            self.thread.join()

def bgr8_to_jpeg(value):
    return bytes(cv2.imencode('.jpg', value)[1])

camera = Camera()
camera.start()

robot = motors.MotorsYukon(mecanum=False)
print("Camera and robot ready.")

## Step 3 — Run the Line Follower

The display shows:
- **Left panel**: live camera feed with prediction overlay
- **Right panel**: yellow HSV mask (useful for debugging)

**To stop the robot safely**, run the emergency stop cell below.

In [3]:
import ipywidgets as widgets
from IPython.display import display
from collections import deque

# Display widgets
display_feed = widgets.Image(format='jpeg', width='45%')
display_mask = widgets.Image(format='jpeg', width='45%')
status_label = widgets.Label(value='Status: Starting...')
layout = widgets.Layout(width='100%')
display(widgets.VBox([
    widgets.HBox([display_feed, display_mask], layout=layout),
    status_label
]))

# ── Yellow HSV range ──────────────────────────────────────────────────────────
YELLOW_LOWER = np.array([20, 100, 100])
YELLOW_UPPER = np.array([35, 255, 255])

# ── Speed / behaviour parameters ─────────────────────────────────────────────
SPEED_FORWARD   = 0.25
SPEED_TURN      = 0.20
SPEED_SEARCH    = 0.25
CONF_THRESHOLD  = 0.55
LOST_FRAMES_MAX = 15

# ── Smoothing buffer ──────────────────────────────────────────────────────────
prediction_buffer = deque(maxlen=3)
lost_frame_count  = 0
last_prediction   = 'forward'
frame_count       = 0

def predict(frame):
    """Run CNN inference on a BGR frame. Returns (class_name, confidence)."""
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    x   = preprocess(rgb).unsqueeze(0).to(device)
    with torch.no_grad():
        out   = model(x)
        probs = torch.softmax(out, dim=1)[0]
    idx = probs.argmax().item()
    return CLASS_NAMES[idx], probs[idx].item()

def yellow_mask_debug(frame):
    """Return a colour-mapped yellow HSV mask for display."""
    hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, YELLOW_LOWER, YELLOW_UPPER)
    return cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)

def on_frame(change):
    global lost_frame_count, last_prediction, frame_count

    frame = change['new']
    if frame is None:
        return

    frame_count += 1

    # ── CNN prediction ────────────────────────────────────────────────────────
    pred, conf = predict(frame)

    # ── Smoothing buffer ──────────────────────────────────────────────────────
    if conf >= CONF_THRESHOLD:
        prediction_buffer.append(pred)
        lost_frame_count = 0
        last_prediction  = pred
    else:
        lost_frame_count += 1

    # Use the most common prediction in the buffer
    if prediction_buffer:
        smoothed_pred = max(set(prediction_buffer), key=prediction_buffer.count)
    else:
        smoothed_pred = 'forward'

    line_lost = lost_frame_count >= LOST_FRAMES_MAX

    # ── Motor commands ────────────────────────────────────────────────────────
    if line_lost:
        if last_prediction == 'left':
            robot.left(SPEED_SEARCH)
        else:
            robot.right(SPEED_SEARCH)
        status = f'SEARCHING (last: {last_prediction}, lost: {lost_frame_count} frames)'
    elif smoothed_pred == 'forward' or conf < CONF_THRESHOLD:
        robot.forward(SPEED_FORWARD)
        status = f'FORWARD  conf={conf:.2f}'
    elif smoothed_pred == 'left':
        robot.left(SPEED_TURN)
        status = f'LEFT     conf={conf:.2f}'
    elif smoothed_pred == 'right':
        robot.right(SPEED_TURN)
        status = f'RIGHT    conf={conf:.2f}'

    status_label.value = f'Status: {status}'

    # ── Display (every 5th frame only) ───────────────────────────────────────
    if frame_count % 5 == 0:
        annotated = frame.copy()
        colour = (0, 255, 0) if not line_lost else (0, 0, 255)
        cv2.putText(annotated, smoothed_pred if not line_lost else 'LOST',
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, colour, 2)
        cv2.putText(annotated, f'conf: {conf:.2f}',
                    (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.7, colour, 2)
        scale = 0.3
        display_feed.value = bgr8_to_jpeg(
            cv2.resize(annotated, None, fx=scale, fy=scale))
        display_mask.value = bgr8_to_jpeg(
            cv2.resize(yellow_mask_debug(frame), None, fx=scale, fy=scale))

camera.observe(on_frame, names=['color_value'])
print("Line follower running. Execute the cell below to stop.")

Line follower running. Execute the cell below to stop.


## ⛔ Emergency Stop
Run this cell at any time to stop the robot and disconnect the camera observer.

In [2]:
camera.unobserve(on_frame, names=['color_value'])
robot.stop()
print("Robot stopped.")


NameError: name 'robot' is not defined

## Step 4 — Fine-Tuning Tips

| Problem | Fix |
|---|---|
| Robot overshoots sharp turns | Reduce `SPEED_TURN` or increase `LOST_FRAMES_MAX` |
| Robot stops too early at turns | Reduce `CONF_THRESHOLD` (e.g. 0.45) |
| Mask shows false yellow detections | Narrow `YELLOW_UPPER[0]` (Hue upper bound) |
| Line not detected outdoors / different lighting | Re-collect data under those conditions |
| Prediction jitters left/right on straights | Collect more `forward` examples near line edges |